In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import shelve
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import *
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
)
from pt_to_api import disjoint_ae, disjoint_ae_learned_sig
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import numpy as np
from torch import nn
from torch import optim
import warnings
from dataclasses import dataclass
from typing import Any
import math
import gc
import pandas as pd
from pt_to_api import benchmark as B
import ast

MODE = "light"
SHELVE_CACHE_ROOT = Path.cwd() / "global-gauss-noise"
SHELVE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]


def generate_synthetic_patches(
    patch_dim=72,
    n_components=10,
    k=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
):
    rng = np.random.RandomState(seed)

    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses at most k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        k_i = rng.randint(1, k + 1)  # active atoms: 1..k
        idx = rng.choice(n_components, k_i, replace=False)
        codes_true[i, idx] = rng.randn(k_i)

    X = codes_true @ W_true

    scale = sigma_x / X.std()
    X *= scale
    W_true *= scale  # keeps codes_true @ W_true ≈ X
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition

# algo0
- the best way might be what i had in mind, clustering and then finding the most similar ones, the algorithm is simple:
- run i for some k runs
    - for each component, calculate how MSE changes with it.  
    - cluster the components, starting with maximum MSE changes.  
    - pick the top MSE changer, with some threshold on the number of repetations required (do not do clusters with 1 object at all)
    - now, the dimensions it explains are good to go, we work without them. 
    - repeat. break when
    - we get zero clusters (many dead atoms compared to previously picked atoms).  
    - The MSE does not change a lot (by some tol).  
    - n_components have been picked.  

In [ ]:
dim = 100
atoms = 20
k = 20
n_samples = 2000
noise_std = 0.2

X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    dim, atoms, k, n_samples=n_samples, noise_std=noise_std
)
scaler = B.MeanPerDimGlobalStdScaler().fit(X)
X_scaled = scaler.transform(X)

In [ ]:
run = B.train(
    X_scaled,
    atoms,
    1e-2,
    epochs=4000,
    baseline_epochs=1000,
    device="mps",
    init_strategy=B.SvdInitStrategy(),
    use_ln_term=False,
)

In [ ]:
B.get_metrics_from_run(run, W_true)

In [ ]:
B.show_closest_component_of_W_for_each_component(
    run.components, W_true, (10, 10), (20, 3), 4
)

In [ ]:
torch.manual_seed(100)
np.random.seed(100)

run2 = B.train(
    X_scaled,
    atoms,
    1e-2,
    epochs=4000,
    baseline_epochs=1000,
    device="mps",
    init_strategy=B.SvdInitStrategy(),
    use_ln_term=False,
)

In [ ]:
# we get extremely good similarities lol, it seems svd finds same stuff generally
# even when it finds lies, it finds the same lies
B.get_metrics_from_run(run2, run.components)

In [ ]:
B.show_closest_component_of_W_for_each_component(
    run2.components, run.components, (10, 10), (20, 3), 4
)

In [ ]:
torch.manual_seed(120)
np.random.seed(120)

run3 = B.train(
    X_scaled,
    atoms,
    1e-2,
    epochs=4000,
    baseline_epochs=1000,
    device="mps",
    use_ln_term=False,
)

In [ ]:
# quite shit similarities
# now im not sure how well they explain the data though.
# all the metrics are fine actually
# is this the right answer too? I'm not sure.
B.get_metrics_from_run(run3, W_true)

In [ ]:
B.show_closest_component_of_W_for_each_component(
    run3.components, W_true, (10, 10), (20, 3), 4
)

Actually, I'm fine with what I have, its okay to not optimise the crap out of this. We'll fix problems as they come. This might not be needed.   
We use SVD init, fast and easy. One extra thing we can do is try with ICA though.  

In [ ]:
run3 = B.train(
    X_scaled,
    atoms,
    1e-2,
    epochs=4000,
    baseline_epochs=1000,
    device="mps",
    use_ln_term=True,
    init_strategy=B.IcaInitStrategy(),
)

In [ ]:
B.show_closest_component_of_W_for_each_component(
    run3.components, W_true, (10, 10), (20, 3), 4
)

In [ ]:
from sklearn.decomposition import FastICA


In [ ]:
# ica_estimator.components_
B.show_closest_component_of_W_for_each_component(
    ica_estimator.components_, W_true, (10, 10), (20, 3), 4
)

In [ ]:
# ica_estimator.components_
dim = 100
atoms = 20
k = 20
n_samples = 2000
noise_std = 0.4

X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    dim, atoms, k, n_samples=n_samples, noise_std=noise_std
)
scaler = B.MeanPerDimGlobalStdScaler().fit(X)
X_scaled = scaler.transform(X)

ica_estimator = FastICA(
    n_components=atoms, max_iter=400, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(X_scaled)
B.show_closest_component_of_W_for_each_component(
    ica_estimator.components_, W_true, (10, 10), (20, 3), 4
)

In [ ]:
import numpy as np

def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]

def generate_synthetic_patches_leaky(
    patch_dim=72,
    n_components=10,
    k=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
    leak=0.0,
):
    rng = np.random.RandomState(seed)
    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))

    # add leakage into non-owned dims
    if leak > 0:
        for i, dims in enumerate(dim_partition):
            other_dims = [d for d in range(patch_dim) if d not in dims]
            W_true[i, other_dims] += leak * rng.randn(len(other_dims))

    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses at most k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        k_i = rng.randint(1, k + 1)
        idx = rng.choice(n_components, k_i, replace=False)
        codes_true[i, idx] = rng.randn(k_i)

    X = codes_true @ W_true
    scale = sigma_x / X.std()
    X *= scale
    W_true *= scale

    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition

In [ ]:
# ica_estimator.components_
dim = 100
atoms = 20
k = 20
n_samples = 2000
noise_std = 0.4

X, W_true, codes_true, dim_partition = generate_synthetic_patches_leaky(
    dim, atoms, k, n_samples=n_samples, noise_std=noise_std, leak=0.2
)
scaler = B.MeanPerDimGlobalStdScaler().fit(X)
X_scaled = scaler.transform(X)

ica_estimator = FastICA(
    n_components=atoms, max_iter=400, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(X_scaled)
B.show_closest_component_of_W_for_each_component(
    ica_estimator.components_, W_true, (10, 10), (20, 3), 4
)

In [ ]:
run3 = B.train(
    X_scaled,
    atoms,
    1e-2,
    epochs=1000,
    baseline_epochs=1000,
    device="mps",
    use_ln_term=True,
    init_strategy=B.IcaInitStrategy(),
)

In [ ]:
B.show_closest_component_of_W_for_each_component(
    run3.components, W_true, (10, 10), (20, 3), 4
)

## Perturbation

In [ ]:
def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]

def generate_synthetic_patches(
    patch_dim=72,
    n_components=10,
    k=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
):
    rng = np.random.RandomState(seed)

    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses at most k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        k_i = rng.randint(1, k + 1)  # active atoms: 1..k
        idx = rng.choice(n_components, k_i, replace=False)
        codes_true[i, idx] = rng.randn(k_i)

    X = codes_true @ W_true

    scale = sigma_x / X.std()
    X *= scale
    W_true *= scale  # keeps codes_true @ W_true ≈ X
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition


def replacement_perturbation(atom_index, ratio, per_sample=False, seed=42):
    def perturb(X, W_true, codes_true, dim_partition):
        rng = np.random.RandomState(seed)
        n_samples, patch_dim = X.shape
        n_affected = int(n_samples * ratio)
        idx = rng.choice(n_samples, n_affected, replace=False)

        dims = dim_partition[atom_index]

        def make_corrupt_atom():
            a = np.zeros(patch_dim)
            a[dims] = rng.randn(len(dims))
            a /= np.linalg.norm(a)
            return a

        if not per_sample:
            corrupt_atom = make_corrupt_atom()

        X_out = X.copy()
        for i in idx:
            coeff = codes_true[i, atom_index]
            if coeff == 0:
                continue
            ca = make_corrupt_atom() if per_sample else corrupt_atom
            X_out[i] -= coeff * W_true[atom_index]
            X_out[i] += coeff * ca

        return X_out, W_true, codes_true, dim_partition
    return perturb

def corrupt_dataset(
    X, W_true, codes_true, dim_partition,
    n_atoms_to_perturb,
    ratio,
    per_sample=False,
    seed=42,
):
    n_components = len(dim_partition)
    rng = np.random.RandomState(seed)
    atoms_to_perturb = rng.choice(n_components, n_atoms_to_perturb, replace=False)

    perturbations = [
        replacement_perturbation(atom_index, ratio, per_sample=per_sample, seed=seed)
        for atom_index in atoms_to_perturb
    ]

    X_corrupted = X
    for perturb in perturbations:
        X_corrupted, W_true, codes_true, dim_partition = perturb(
            X_corrupted, W_true, codes_true, dim_partition
        )

    return X_corrupted, W_true, codes_true, dim_partition, atoms_to_perturb


def generate_corrupted_dataset(
    n_atoms_to_perturb,
    ratio,
    per_sample=False,
    patch_dim=72,
    n_components=10,
    k=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
):
    X, W_true, codes_true, dim_partition = generate_synthetic_patches(
        patch_dim=patch_dim,
        n_components=n_components,
        k=k,
        n_samples=n_samples,
        noise_std=noise_std,
        seed=seed,
        sigma_x=sigma_x,
    )

    return corrupt_dataset(
        X, W_true, codes_true, dim_partition,
        n_atoms_to_perturb=n_atoms_to_perturb,
        ratio=ratio,
        per_sample=per_sample,
        seed=seed,
    )

In [ ]:

X_corrupted, W_true, codes_true, dim_partition, atoms_to_perturb = generate_corrupted_dataset(5, 0.2, True, 100, 10, 10, 1000, 0.01)

In [ ]:
scaler = B.MeanPerDimGlobalStdScaler().fit(X_corrupted)
X_scaled = scaler.transform(X_corrupted)

ica_estimator = FastICA(
    n_components=atoms, max_iter=400, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(X_scaled)
B.show_closest_component_of_W_for_each_component(
    ica_estimator.components_, W_true, (10, 10), (20, 3), 4
)

In [ ]:
run3 = B.train(
    X_scaled,
    atoms,
    1e-2,
    epochs=1000,
    baseline_epochs=1000,
    device="mps",
    use_ln_term=True,
    init_strategy=B.SvdInitStrategy(),
)
B.show_closest_component_of_W_for_each_component(
    run3.components, W_true, (10, 10), (20, 3), 4
)